<a href="https://colab.research.google.com/github/faisalkhan6ai-sketch/AI-ML-Internship-Tasks/blob/main/End-to-End%20ML%20Pipeline%20with%20Scikit-learn%20Pipeline%20API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 296)

In [20]:
R"""
End-to-End ML Pipeline for Customer Churn Prediction
Dataset: Telco Customer Churn
Author: Production-Ready Pipeline Demo
"""

# ───────────────────────────────────
# 0. Imports
# ───────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import io
import os

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    f1_score, precision_score, recall_score
)

# ───────────────────────────────────
# 1. Load / Generate Telco-style Churn Dataset
# ───────────────────────────────────
def load_telco_dataset(file_path=None):
    """
    Generates a realistic Telco-style churn dataset or loads from CSV.
    If file_path is provided, loads from CSV. Otherwise, generates synthetic data.
    """
    if file_path:
        df = pd.read_csv(file_path)
        # Rename 'customerID' if it exists and drop it, not used in this pipeline
        if 'customerID' in df.columns: # Changed from df.columns[0] == 'customerID'
            df = df.drop(columns=['customerID'])
        # Convert 'TotalCharges' to numeric, coercing errors to NaN
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        # Convert 'SeniorCitizen' to 'Yes'/'No' for consistency with synthetic data logic, if it's 0/1
        if df['SeniorCitizen'].dtype == 'int64':
            df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'})

        return df

    np.random.seed(42)
    n = 7043

    gender          = np.random.choice(["Male", "Female"], n)
    senior          = np.random.choice([0, 1], n, p=[0.84, 0.16])
    partner         = np.random.choice(["Yes", "No"], n)
    dependents      = np.random.choice(["Yes", "No"], n, p=[0.30, 0.70])
    tenure          = np.random.randint(0, 72, n)
    phone_service   = np.random.choice(["Yes", "No"], n, p=[0.90, 0.10])
    multiple_lines  = np.where(
        phone_service == "No", "No phone service",
        np.random.choice(["Yes", "No"], n)
    )
    internet_service = np.random.choice(
        ["DSL", "Fiber optic", "No"], n, p=[0.34, 0.44, 0.22]
    )
    online_security = np.where(
        internet_service == "No", "No internet service",
        np.random.choice(["Yes", "No"], n)
    )
    tech_support = np.where(
        internet_service == "No", "No internet service",
        np.random.choice(["Yes", "No"], n)
    )
    contract        = np.random.choice(
        ["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.21, 0.24]
    )
    paperless       = np.random.choice(["Yes", "No"], n)
    payment_method  = np.random.choice(
        ["Electronic check", "Mailed check",
         "Bank transfer (automatic)", "Credit card (automatic)"], n
    )
    monthly_charges = np.round(np.random.uniform(18, 119, n), 2)
    total_charges_raw = tenure * monthly_charges + np.random.normal(0, 50, n)
    total_charges   = np.where(
        total_charges_raw < 0, np.nan,
        np.round(total_charges_raw, 2)
    )

    # Churn probability influenced by contract type, tenure, internet service
    churn_prob = (
        0.05
        + 0.25 * (contract == "Month-to-month")
        - 0.003 * tenure
        + 0.10 * (internet_service == "Fiber optic")
        + 0.05 * (senior == 1)
    )
    churn_prob = np.clip(churn_prob, 0.03, 0.75)
    churn = np.random.binomial(1, churn_prob).astype(str)
    churn = np.where(churn == "1", "Yes", "No")

    df = pd.DataFrame({
        "gender": gender,
        "SeniorCitizen": senior,
        "Partner": partner,
        "Dependents": dependents,
        "tenure": tenure,
        "PhoneService": phone_service,
        "MultipleLines": multiple_lines,
        "InternetService": internet_service,
        "OnlineSecurity": online_security,
        "TechSupport": tech_support,
        "Contract": contract,
        "PaperlessBilling": paperless,
        "PaymentMethod": payment_method,
        "MonthlyCharges": monthly_charges,
        "TotalCharges": total_charges,
        "Churn": churn,
    })
    return df

# ───────────────────────────────────
# 2. Exploratory Data Analysis (EDA)
# ───────────────────────────────────
def run_eda(df):
    print("\n" + "="*60)
    print("  EXPLORATORY DATA ANALYSIS")
    print("="*60)
    print(f"\n📊 Dataset Shape : {df.shape}")
    print(f"   Rows          : {df.shape[0]:,}")
    print(f"   Columns       : {df.shape[1]}")

    print("\n📋 Column Types:")
    print(df.dtypes.to_string())

    print(f"\n❓ Missing Values:")
    missing = df.isnull().sum()
    print(missing[missing > 0].to_string() if missing.any() else "   None found ✓")

    print(f"\n🎯 Churn Distribution:")
    churn_counts = df["Churn"].value_counts()
    for label, count in churn_counts.items():
        pct = count / len(df) * 100
        print(f"   {label}: {count:,}  ({pct:.1f}%)")

    print(f"\n📈 Numeric Feature Summary:")
    print(df.describe().round(2).to_string())

# ───────────────────────────────────
# 3. Preprocessing
# ───────────────────────────────────
def preprocess_data(df):
    """Clean and split data; return X, y."""
    df = df.copy()

    # Fix TotalCharges (coerce errors → NaN handled by imputer)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Target encoding
    df["Churn"] = (df["Churn"] == "Yes").astype(int)

    X = df.drop(columns=["Churn"])
    y = df["Churn"]
    return X, y

def get_feature_groups(X):
    """Return numeric and categorical column lists."""
    numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    return numeric_cols, categorical_cols

# ───────────────────────────────────
# 4. Build Preprocessing Pipeline
# ───────────────────────────────────
def build_preprocessor(numeric_cols, categorical_cols):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer,  numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ])
    return preprocessor

# ───────────────────────────────────
# 5. Build Full Model Pipelines
# ───────────────────────────────────
def build_pipelines(preprocessor):
    lr_pipeline = Pipeline(steps=[
        ("preprocessor",       preprocessor),
        ("logistic_regression", LogisticRegression(
            max_iter=1000, random_state=42, class_weight="balanced"
        )),
    ])

    rf_pipeline = Pipeline(steps=[
        ("preprocessor",    preprocessor),
        ("random_forest",   RandomForestClassifier(
            random_state=42, class_weight="balanced", n_jobs=-1
        )),
    ])

    return lr_pipeline, rf_pipeline

# ───────────────────────────────────
# 6. GridSearchCV Hyperparameter Tuning
# ───────────────────────────────────
def tune_logistic_regression(pipeline, X_train, y_train):
    print("\n" + "="*60)
    print("  LOGISTIC REGRESSION — GridSearchCV")
    print("="*60)

    param_grid = {
        "logistic_regression__C":        [0.01, 0.1, 1.0, 10.0],
        "logistic_regression__solver":   ["liblinear", "lbfgs"],
        "logistic_regression__penalty":  ["l2"],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        pipeline, param_grid,
        cv=cv, scoring="roc_auc",
        n_jobs=-1, verbose=0, refit=True
    )
    grid_search.fit(X_train, y_train)

    print(f"\n✅ Best Parameters : {grid_search.best_params_}")
    print(f"   Best CV ROC-AUC : {grid_search.best_score_:.4f}")
    return grid_search.best_estimator_

def tune_random_forest(pipeline, X_train, y_train):
    print("\n" + "="*60)
    print("  RANDOM FOREST — GridSearchCV")
    print("="*60)

    param_grid = {
        "random_forest__n_estimators": [100, 200],
        "random_forest__max_depth":    [None, 10, 20],
        "random_forest__min_samples_split": [2, 5],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        pipeline, param_grid,
        cv=cv, scoring="roc_auc",
        n_jobs=-1, verbose=0, refit=True
    )
    grid_search.fit(X_train, y_train)

    print(f"\n✅ Best Parameters : {grid_search.best_params_}")
    print(f"   Best CV ROC-AUC : {grid_search.best_score_:.4f}")
    return grid_search.best_estimator_

# ───────────────────────────────────
# 7. Evaluation
# ───────────────────────────────────
def evaluate_model(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred, zero_division=0),
        "F1-Score":  f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_test, y_proba),
    }

    print(f"\n{'─'*50}")
    print(f"  {name} — Test Set Metrics")
    print(f"{'─'*50}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.4f}")

    print(f"\n  Classification Report:\n")
    print(classification_report(y_test, y_pred,
                                target_names=["No Churn", "Churn"]))
    return metrics, y_pred, y_proba

# ───────────────────────────────────
# 8. Visualisations
# ───────────────────────────────────
def plot_all(df, best_lr, best_rf, X_test, y_test,
             lr_metrics, rf_metrics,
             lr_proba, rf_proba, numeric_cols):

    fig = plt.figure(figsize=(22, 24))
    fig.patch.set_facecolor("#0F1117")
    colors = {"primary": "#7C3AED", "secondary": "#06B6D4",
              "accent": "#F59E0B", "success": "#10B981",
              "danger": "#EF4444", "bg": "#1E2130", "text": "#E2E8F0"}

    def ax_style(ax, title):
        ax.set_facecolor(colors["bg"])
        for s in ax.spines.values():
            s.set_color("#374151")
        ax.tick_params(colors=colors["text"], labelsize=9)
        ax.set_title(title, color=colors["text"], fontsize=11,
                     fontweight="bold", pad=10)
        ax.xaxis.label.set_color(colors["text"])
        ax.yaxis.label.set_color(colors["text"])

    axes = [fig.add_subplot(4, 3, i+1) for i in range(12)]

    # ── 1. Churn Distribution ──
    ax = axes[0]
    churn_counts = df["Churn"].map({0: "No Churn", 1: "Churn"}).value_counts()
    bars = ax.bar(churn_counts.index, churn_counts.values,
                  color=[colors["success"], colors["danger"]], width=0.5,
                  edgecolor="#374151")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{int(bar.get_height()):,}', ha="center", color=colors["text"],
                fontsize=9, fontweight="bold")
    ax_style(ax, "❶ Churn Distribution")

    # ── 2. Churn by Contract Type ──
    ax = axes[1]
    ct = pd.crosstab(df["Contract"], df["Churn"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="bar", ax=ax,
                color=[colors["success"], colors["danger"]],
                edgecolor="#374151", width=0.6)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.set_ylabel("Percentage (%)")
    ax.legend(["No Churn", "Churn"], facecolor="#374151",
              labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "❷ Churn by Contract Type")

    # ── 3. Monthly Charges Distribution ──
    ax = axes[2]
    no_churn = df[df["Churn"]==0]["MonthlyCharges"]
    churn    = df[df["Churn"]==1]["MonthlyCharges"]
    ax.hist(no_churn, bins=30, alpha=0.7, color=colors["success"], label="No Churn")
    ax.hist(churn,    bins=30, alpha=0.7, color=colors["danger"],  label="Churn")
    ax.set_xlabel("Monthly Charges ($")
    ax.set_ylabel("Count")
    ax.legend(facecolor="#374151", labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "❸ Monthly Charges by Churn")

    # ── 4. Tenure Distribution ──
    ax = axes[3]
    ax.hist(df[df["Churn"]==0]["tenure"], bins=30, alpha=0.7,
            color=colors["primary"], label="No Churn")
    ax.hist(df[df["Churn"]==1]["tenure"], bins=30, alpha=0.7,
            color=colors["accent"],  label="Churn")
    ax.set_xlabel("Tenure (months)")
    ax.set_ylabel("Count")
    ax.legend(facecolor="#374151", labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "❹ Tenure Distribution by Churn")

    # ── 5. Correlation Heatmap (numeric) ──
    ax = axes[4]
    num_df = df[numeric_cols + ["Churn"]].copy()
    num_df["Churn"] = pd.to_numeric(num_df["Churn"], errors="coerce")
    corr = num_df.corr(numeric_only=True)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    cmap = sns.diverging_palette(250, 15, s=75, l=40, n=9, as_cmap=True)
    sns.heatmap(corr, ax=ax, mask=mask, cmap=cmap, center=0,
                annot=True, fmt=".2f", annot_kws={"size": 8},
                linewidths=0.5, linecolor="#374151",
                cbar_kws={"shrink": 0.8})
    ax_style(ax, "❺ Correlation Heatmap")

    # ── 6. Churn by Internet Service ──
    ax = axes[5]
    df_int = df.copy()
    df_int["Churn"] = pd.to_numeric(df_int["Churn"], errors="coerce")
    internet_churn = df_int.groupby("InternetService")["Churn"].mean() * 100
    bars = ax.bar(internet_churn.index, internet_churn.values,
                  color=[colors["primary"], colors["secondary"], colors["accent"]],
                  edgecolor="#374151", width=0.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha="center",
                color=colors["text"], fontsize=9)
    ax.set_ylabel("Churn Rate (%)")
    ax_style(ax, "❻ Churn Rate by Internet Service")

    # ── 7. Confusion Matrix — LR ──
    ax = axes[6]
    cm = confusion_matrix(y_test, best_lr.predict(X_test))
    sns.heatmap(cm, ax=ax, annot=True, fmt="d", cmap="Purples",
                cbar=False, linewidths=1, linecolor="#374151",
                xticklabels=["Predicted\nNo Churn", "Predicted\nChurn"],
                yticklabels=["Actual\nNo Churn", "Actual\nChurn"])
    ax_style(ax, "❼ Confusion Matrix — Logistic Regression")

    # ── 8. Confusion Matrix — RF ──
    ax = axes[7]
    cm_rf = confusion_matrix(y_test, best_rf.predict(X_test))
    sns.heatmap(cm_rf, ax=ax, annot=True, fmt="d", cmap="Greens",
                cbar=False, linewidths=1, linecolor="#374151",
                xticklabels=["Predicted\nNo Churn", "Predicted\nChurn"],
                yticklabels=["Actual\nNo Churn", "Actual\nChurn"])
    ax_style(ax, "❽ Confusion Matrix — Random Forest")

    # ── 9. ROC Curves ──
    ax = axes[8]
    for name, proba, col in [
        ("Logistic Regression", lr_proba, colors["primary"]),
        ("Random Forest",       rf_proba, colors["secondary"]),
    ]:
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, color=col, lw=2, label=f"{name} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "w--", lw=1, alpha=0.4)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(facecolor="#374151", labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "❾ ROC Curves Comparison")

    # ── 10. Feature Importance — RF ──
    ax = axes[9]
    rf_model = best_rf.named_steps["random_forest"]
    preprocessor = best_rf.named_steps["preprocessor"]
    feature_names = (
        list(preprocessor.transformers_[0][2]) +  # numeric
        list(preprocessor.transformers_[1][1]
             .named_steps["onehot"]
             .get_feature_names_out(
                 preprocessor.transformers_[1][2]))  # one-hot
    )
    importances = rf_model.feature_importances_
    top_n = 12
    idx = np.argsort(importances)[-top_n:]
    ax.barh([feature_names[i] for i in idx], importances[idx],
            color=colors["secondary"], edgecolor="#374151")
    ax_style(ax, "❿ Top Feature Importances (RF)")

    # ── 11. Model Metrics Comparison Bar ──
    ax = axes[10]
    metric_names = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
    lr_vals = [lr_metrics[m] for m in metric_names]
    rf_vals = [rf_metrics[m] for m in metric_names]
    x = np.arange(len(metric_names))
    w = 0.35
    ax.bar(x - w/2, lr_vals, width=w, label="Logistic Regression",
           color=colors["primary"], edgecolor="#374151")
    ax.bar(x + w/2, rf_vals, width=w, label="Random Forest",
           color=colors["secondary"], edgecolor="#374151")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names, rotation=15, ha="right", fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.legend(facecolor="#374151", labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "➀ Model Performance Comparison")

    # ── 12. Prediction Probability Distribution ──
    ax = axes[11]
    ax.hist(lr_proba[y_test == 0], bins=30, alpha=0.6,
            color=colors["success"], label="True No Churn")
    ax.hist(lr_proba[y_test == 1], bins=30, alpha=0.6,
            color=colors["danger"], label="True Churn")
    ax.axvline(0.5, color="white", linestyle="--", lw=1.5, label="Threshold=0.5")
    ax.set_xlabel("Predicted Probability of Churn")
    ax.set_ylabel("Count")
    ax.legend(facecolor="#374151", labelcolor=colors["text"], fontsize=8)
    ax_style(ax, "➁ Probability Distribution (LR)")

    fig.suptitle("🔮 Customer Churn Prediction — End-to-End ML Pipeline",
                 fontsize=16, fontweight="bold",
                 color=colors["text"], y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    out = "/mnt/user-data/outputs/churn_pipeline_report.png"
    plt.savefig(out, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"\n📊 Dashboard saved → {out}")

# ───────────────────────────────────
# 9. Export Pipelines with joblib
# ───────────────────────────────────
def export_pipelines(best_lr, best_rf):
    lr_path = "/mnt/user-data/outputs/lr_churn_pipeline.joblib"
    rf_path = "/mnt/user-data/outputs/rf_churn_pipeline.joblib"
    joblib.dump(best_lr, lr_path, compress=3)
    joblib.dump(best_rf, rf_path, compress=3)
    print(f"\n💾 Exported → {lr_path}")
    print(f"   Exported → {rf_path}")
    return lr_path, rf_path

def verify_export(lr_path, rf_path, X_test):
    """Load saved pipelines and confirm they still predict."""
    lr_loaded = joblib.load(lr_path)
    rf_loaded = joblib.load(rf_path)
    lr_preds = lr_loaded.predict(X_test[:5])
    rf_preds = rf_loaded.predict(X_test[:5])
    print("\n✅ Export Verification:")
    print(f"   LR predictions (first 5) : {lr_preds}")
    print(f"   RF predictions (first 5) : {rf_preds}")

# ───────────────────────────────────
# 10. MAIN
# ───────────────────────────────────
def main():
    print("="*60)
    print("  TELCO CHURN — END-TO-END ML PIPELINE")
    print("="*60)

    # Ensure the output directory exists
    os.makedirs("/mnt/user-data/outputs/", exist_ok=True)

    # Load
    df = load_telco_dataset("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    run_eda(df)

    # Preprocess
    X, y = preprocess_data(df)
    numeric_cols, categorical_cols = get_feature_groups(X)
    print(f"\n🔢 Numeric features    : {numeric_cols}")
    print(f"🔠 Categorical features: {categorical_cols}")

    # Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    print(f"\n📦 Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")

    # Build preprocessor & pipelines
    preprocessor = build_preprocessor(numeric_cols, categorical_cols)
    lr_pipeline, rf_pipeline = build_pipelines(preprocessor)

    # Tune
    print("\n⚙️  Running GridSearchCV (this may take ~1 minute)…")
    best_lr = tune_logistic_regression(lr_pipeline, X_train, y_train)
    best_rf = tune_random_forest(rf_pipeline, X_train, y_train)

    # Evaluate
    print("\n" + "="*60)
    print("  TEST SET EVALUATION")
    print("="*60)
    lr_metrics, lr_pred, lr_proba = evaluate_model(
        "Logistic Regression", best_lr, X_test, y_test)
    rf_metrics, rf_pred, rf_proba = evaluate_model(
        "Random Forest",       best_rf, X_test, y_test)

    # Winner
    winner = "Logistic Regression" if lr_metrics["ROC-AUC"] > rf_metrics["ROC-AUC"] \
             else "Random Forest"
    print(f"\n🏆 Best Model by ROC-AUC: {winner}")

    # Cross-validation on best model
    best_model = best_lr if winner == "Logistic Regression" else best_rf
    cv_scores = cross_val_score(best_model, X, y, cv=5,
                                scoring="roc_auc", n_jobs=-1)
    print(f"\n📊 5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Plot
    # Pass the y as int for df
    df_plot = df.copy()
    df_plot["Churn"] = (df_plot["Churn"] == "Yes").astype(int) \
                       if df_plot["Churn"].dtype == object else df_plot["Churn"]
    plot_all(df_plot, best_lr, best_rf, X_test, y_test,
             lr_metrics, rf_metrics, lr_proba, rf_proba, numeric_cols)

    # Export
    lr_path, rf_path = export_pipelines(best_lr, best_rf)
    verify_export(lr_path, rf_path, X_test)

    print("\n" + "="*60)
    print("  PIPELINE COMPLETE ✅")
    print("="*60)

if __name__ == "__main__":
    main()

  TELCO CHURN — END-TO-END ML PIPELINE

  EXPLORATORY DATA ANALYSIS

📊 Dataset Shape : (7043, 20)
   Rows          : 7,043
   Columns       : 20

📋 Column Types:
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object

❓ Missing Values:
TotalCharges    11

🎯 Churn Distribution:
   No: 5,174  (73.5%)
   Yes: 1,869  (26.5%)

📈 Numeric Feature Summary:
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7032.00
mean     32.37           64.76       2283.30
std      